# 1. Read CSV

In [34]:
import pandas as pd


df = pd.read_csv('data/dataset.csv')

display(df.head())

,order,year,country,study_focus,historical_site_type,historical_site_type_sub,platform,device,technique,technique_sub,software_data,software_modeling,software_render
0,1,2015,South Korea,Visualization,Building,Religious,VR,HMD,3D Scanning;Modeling & Reconstruction,Laser Scanning;3D Modeling,Australis Photometric,Autodesk 3ds Max,Unity
1,2,2015,Spain,Reconstruction,Archaeological Site,LandBased,AR,Mobile,Image-Based Techniques;Modeling & Reconstruction,Photogrammetry;Structure from Motion (SfM);3D ...,Agisoft Metashape,Blender,NaN
2,3,2015,Peru,Visualization,Archaeological Site,LandBased,AR,Mobile,Image-Based Techniques,Photogrammetry;Structure from Motion (SfM),Agisoft Metashape,NaN,NaN
3,4,2015,Greece,Reconstruction,Archaeological Site,LandBased,AR;MR,HMD;Mobile,Image-Based Techniques;Geospatial Techniques,Photogrammetry;Structure from Motion (SfM);Geo...,Agisoft Metashape,Blender;Autodesk 3ds Max,Unity
4,5,2015,Italy,Reconstruction,Archaeological Site,LandBased,VR,HMD;Mobile,Image-Based Techniques;Geospatial Techniques;M...,Photogrammetry;Geographic Information System (...,QGIS,Autodesk Maya,Unity


# 2. Add ZELEMÉR (#116)

In [35]:
my_project = {
    'order': 116,
    'year': 2026,
    'country': 'Hungary',
    'study_focus': 'Reconstruction',
    'historical_site_type': 'Building',
    'historical_site_type_sub': 'Religious',
    'platform': 'VR',
    'device': 'Mobile',
    'technique': 'Image-Based Techniques;Modeling & Reconstruction;Data Processing',
    'technique_sub': 'Image-Based Modeling (IBM);3D Modeling;Archaeological Interpretation',
    'software_data': '',
    'software_modeling': 'Autodesk AutoCAD;SketchUp',
    'software_render': 'Lumion'
}

df.loc[len(df)] = my_project

# 3. Tokenize dataset

In [36]:
def get_multi_value_columns(df):
    multi_value_cols = []
    
    for col in df.columns:
        if df[col].dropna().astype(str).str.contains(";").any():
            multi_value_cols.append(col)
            
    return multi_value_cols


def tokenize_columns(df, columns_to_tokenize):
    for col in columns_to_tokenize:
        df[col] = df[col].fillna("").apply(
            lambda x: [item.strip() for item in str(x).split(";") if item.strip()]
        )
        
    return df

In [37]:
multi_value_columns = get_multi_value_columns(df)

print(multi_value_columns)

['platform', 'device', 'technique', 'technique_sub', 'software_data', 'software_modeling', 'software_render']


In [38]:
df_added = tokenize_columns(df, multi_value_columns)

display(df_added.tail(3))

,order,year,country,study_focus,historical_site_type,historical_site_type_sub,platform,device,technique,technique_sub,software_data,software_modeling,software_render
113,114,2025,Portugal,Restoration,Archaeological Site,LandBased,[VR],[HMD],"[Image-Based Techniques, Modeling & Reconstruc...","[Photogrammetry, 3D Modeling]",[Agisoft Metashape],[Blender],[Unity]
114,115,2025,Germany,Reconstruction,Building,Fortification,[VR],[HMD],"[Image-Based Techniques, Modeling & Reconstruc...","[Photogrammetry, 3D Modeling]",[Agisoft Metashape],[Meshlab],[Unity]
115,116,2026,Hungary,Reconstruction,Building,Religious,[VR],[Mobile],"[Image-Based Techniques, Modeling & Reconstruc...","[Image-Based Modeling (IBM), 3D Modeling, Arch...",[],"[Autodesk AutoCAD, SketchUp]",[Lumion]


# 4. Add ÉLŐ scores

In [39]:
ELOS =  [
    1349, 1581, 1544, 1505, 1596, 1231, 1393, 1198, 1232, 1646, 
    1377, 1550, 1492, 1577, 1286, 1199, 1771, 1669, 1320, 1441, 
    1431, 1520, 1574, 1426, 1304, 1401, 1305, 1683, 1272, 1428, 
    1488, 1691, 1908, 1556, 1406, 1509, 1379, 1557, 1471, 1751, 
    1775, 1491, 1463, 1386, 1633, 1566, 1497, 1330, 1497, 1408, 
    1461, 1812, 1298, 1388, 1626, 1657, 1472, 1337, 1483, 1491, 
    1559, 1310, 1522, 1442, 1421, 1575, 1670, 1619, 1288, 1364, 
    1655, 1728, 1442, 1381, 1537, 1493, 1464, 1460, 1665, 1388, 
    1741, 1643, 1603, 1285, 1537, 1687, 1544, 1449, 1581, 1537, 
    1673, 1497, 1434, 1585, 1579, 1450, 1263, 1578, 1568, 1425, 
    1663, 1477, 1233, 1448, 1460, 1342, 1674, 1545, 1600, 1266, 
    1608, 1742, 1596, 1446, 1708, 1439
]

df_added['elo'] = ELOS

print(f"Data shape: {df_added.shape}")
display(df_added[['order', 'elo']])

Data shape: (116, 14)


,order,elo
0,1,1349
1,2,1581
2,3,1544
3,4,1505
4,5,1596
...,...,...
111,112,1742
112,113,1596
113,114,1446
114,115,1708


# 5. Summary

In [40]:
def get_column_statistics(df, col_name, is_list):
    total_rows = len(df)
    stats = []
    
    if is_list:
        all_items = []
        for lst in df[col_name].dropna():
            if isinstance(lst, list):
                all_items.extend(lst)
        unique_labels = sorted(list(set(all_items)))
    else:
        unique_labels = sorted(df[col_name].dropna().unique().tolist())
        
    for label in unique_labels:
        if is_list:
            count = df[col_name].apply(lambda x: label in x if isinstance(x, list) else False).sum()
        else:
            count = (df[col_name] == label).sum()
            
        percent = round((count / total_rows) * 100, 1)
        stats.append((label, count, percent))
        
    return stats


def print_formatted_table(col_name, stats, w_name, w_count, w_pct):
    print("\n")
    print(f"{col_name:<{w_name}} | {'count':<{w_count}} | {'percent':<{w_pct}}")
    print("─" * 70)

    for label, count, percent in stats:
        percent_str = f"{percent}%"
        print(f"{str(label):<{w_name}} | {str(count):<{w_count}} | {percent_str:<{w_pct}}")


def print_dataset_summary(df, multi_value_cols, w_name=40, w_count=10, w_pct=10):
    ignore_cols = ['order', 'year', 'country', 'elo']
        
    for col in df.columns:
        if col in ignore_cols:
            continue
            
        is_list = col in multi_value_cols
        stats = get_column_statistics(df, col, is_list)
        print_formatted_table(col, stats, w_name, w_count, w_pct)


print_dataset_summary(df_added, multi_value_columns)



study_focus                              | count      | percent   
──────────────────────────────────────────────────────────────────────
Reconstruction                           | 48         | 41.4%     
Restoration                              | 9          | 7.8%      
Visualization                            | 59         | 50.9%     


historical_site_type                     | count      | percent   
──────────────────────────────────────────────────────────────────────
Archaeological Site                      | 55         | 47.4%     
Artistic Feature                         | 6          | 5.2%      
Building                                 | 49         | 42.2%     
Natural Space                            | 6          | 5.2%      


historical_site_type_sub                 | count      | percent   
──────────────────────────────────────────────────────────────────────
ArchitecturalAsset                       | 2          | 1.7%      
Artifact                                 | 4